# Smoke test — `biokg` — one anonymized example
Validates the full pipeline on **biokg**: KG load → evidence dossier → drug-name anonymization → model call → JSON parse.

Run this notebook from inside the repo (it auto-finds `config.yaml`). Pick the model in the next cell; its key must be in `../.env`.

In [1]:
import os
# Model to smoke-test (key must be in .env). Swap as needed:
os.environ['BKG_MODELS'] = 'groq:llama-3.3-70b-versatile'   # or 'gemini:gemini-2.0-flash' / 'groq:llama-3.3-70b-versatile'
KG_TO_TEST = 'biokg'


In [2]:
# Bootstrap: exec the harness setup cells from 09 (paths, config, prompt, parse, dispatch, kg-block, queries)
import json, os
_p = os.path.join(os.path.dirname(os.getcwd()), 'eval_notebooks', '09_llm_integration.ipynb')
_p = _p if os.path.exists(_p) else 'eval_notebooks/09_llm_integration.ipynb'
_p = _p if os.path.exists(_p) else '09_llm_integration.ipynb'
_nb = json.load(open(_p))
for _i in [2, 4, 6, 8, 10, 12, 14, 16]:
    exec(''.join(_nb['cells'][_i]['source']), globals())
KGS = [KG_TO_TEST]   # restrict to the single KG under test
print('setup OK | model:', MODELS, '| KG:', KGS, '| anonymize:', ANONYMIZE, '| n_queries:', len(QUERIES))


repo root: /Users/shil6661/biokgsuite
MODE=real  models=['groq:llama-3.3-70b-versatile']
GOLD=gold_standard_v3_common3.tsv  KGS=['primekg', 'drkg', 'biokg']  n_diseases=76  seeds=[0]  shuffles=2
49 queries  (pool=8):
  - [CNS] Chorea associated with Huntington's disease (DOID:12858)  true drug: valbenazine
  - [Cardiovascular] Chronic kidney disease (DOID:784)  true drug: empagliflozin
  - [Hematology] Polycythaemia vera and essential thrombocytopenia (DOID:8997)  true drug: peginterferon alfa-2a
  - [Immunology] Generalized myasthenia gravis (DOID:437)  true drug: inebilizumab
  - [Infectious] Acute HCV pediatric (DOID:684)  true drug: glecaprevir/pibrentasvir
  - [Metabolic] PKU expanded to ≥12yr (DOID:9281)  true drug: pegvaliase-pqpz
  - [Nephrology] Nephrotic syndrome (DOID:1184)  true drug: rituximab
  - [Oncology] Follicular lymphoma (DOID:0050873)  true drug: tafasitamab
  - [Ophthalmology] Retinopathy of prematurity (DOID:13025)  true drug: aflibercept
  - [CNS] Agitation asso

In [3]:
# Build ONE anonymized example for this KG (prefer a disease the KG actually covers)
import numpy as np
block_for, disease_profile_fn = make_kg_block_fn(KG_TO_TEST, bridge_mode=BRIDGE_MODE, cap=CAP,
                                                 anonymize=ANONYMIZE, anonymize_genes=ANONYMIZE_GENES)
model = MODELS[0]
def _build(q):
    ids = sorted({c['drug_id'] for c in q['candidates']})
    cmap = {d: f'Drug-{i+1}' for i, d in enumerate(ids)} if ANONYMIZE else None
    rng = np.random.default_rng(SEED)
    view, pos = assign_letters_and_evidence(q['candidates'], block_for, q['disease_id'], 'kg', MOCK, rng, code_map=cmap)
    return view, pos
chosen = None
for q in QUERIES:
    view, pos = _build(q)
    if any(c['letter'] == pos and c.get('evidence') for c in view):
        chosen = (q, view, pos); break
if chosen is None:
    q = QUERIES[0]; view, pos = _build(q); chosen = (q, view, pos)
    print('(note: no KG evidence found for the true drug in any query — showing first query anyway)')
q, view, pos = chosen
dprof = disease_profile_fn(q['disease_id'], q['disease_name'])
prompt = build_prompt(q['disease_name'], q['disease_id'], view, disease_profile=dprof)
print('='*72)
print('KG          :', KG_TO_TEST)
print('DISEASE     :', q['disease_name'], '(', q['disease_id'], ')')
print('TRUE drug is hidden as:', pos, '  (drug names anonymized as Drug-1..8)')
print('='*72)
print(prompt)


KG          : biokg
DISEASE     : Chorea associated with Huntington's disease ( DOID:12858 )
TRUE drug is hidden as: F   (drug names anonymized as Drug-1..8)
You are assisting a drug repurposing scientist. For the target disease below,
you are given candidate drugs. Some include evidence retrieved from a
biomedical knowledge graph, reported as mechanistic links
(drug -> target -> pathway -> disease) and related annotations.


Target disease: Chorea associated with Huntington's disease (DOID:12858)
Disease profile (from the knowledge graph): Chorea associated with Huntington's disease is associated with gene IP6K2; Chorea associated with Huntington's disease is associated with gene IP6K2; Chorea associated with Huntington's disease is associated with gene S29A1; Chorea associated with Huntington's disease is associated with gene S29A1; Chorea associated with Huntington's disease is associated with gene FAAH1; Chorea associated with Huntington's disease is associated with gene DBLOH; Cho

In [5]:
import sys; print(sys.executable)

/opt/homebrew/opt/python@3.11/bin/python3.11


In [4]:
# Single model call + parse
resp = rank_call(model, prompt, view, 0)
print('RAW RESPONSE  (', model, ')\n' + '-'*72)
print(resp)
print('-'*72)
is_err = isinstance(resp, str) and resp.startswith('__ERROR__')
ordered, fields = parse_response(resp, [c['letter'] for c in view])
print('parsed ranking :', ordered)
print('true-drug rank :', rank_of(pos, ordered, POOL_SIZE), '/', POOL_SIZE)
ok = (not is_err) and bool(ordered) and len(ordered) == len(view)
print('\nSMOKE TEST PASSED \u2705  (KG loaded, evidence built, anonymized, full ranking parsed)' if ok
      else '\nSMOKE TEST FAILED \u274c  — API error or incomplete parse; see raw response above')


RAW RESPONSE  ( groq:llama-3.3-70b-versatile )
------------------------------------------------------------------------
{
  "reasoning": "The knowledge graph provides some associations between genes and the target disease, but most candidate drugs do not have direct links to these genes or the disease. Therefore, prior pharmacological knowledge is used to assess the candidates, and confidence is generally low due to the lack of direct evidence. Conflicts between the knowledge graph and prior knowledge are noted where applicable.",
   "ranking": [
       {
           "rank": 1,
           "drug": "D",
           "basis": "prior",
           "evidence_agreement": "insufficient",
           "confidence": 2,
           "rationale": "Drug-7 targets multiple inflammatory mediators, which could potentially modulate the neuroinflammatory component of Huntington's disease, although direct evidence is lacking."
       },
       {
           "rank": 2,
           "drug": "C",
           "basis": 